In [12]:
!pip install gdown
!pip install py-vncorenlp
!pip install vncorenlp
!pip install rouge-score
!pip install transformers
!pip install sacrebleu


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.6 MB/s eta 0:00:0000:010:01
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=364fd02b19722f76de2a8b76a23895cf95a71a2389c2ef7cc08eeca181c56920
  Stored in directory: /root/.cache/pip/wheels/6d/2d/d6/158260bfd6820d144535857b80cc112bee5c3aa6d81b6dc049
Successfully built py-vncorenlp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.7 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=388bc2f7739c25b491e0b329a8a93daff37b9a2ceaf68ada98d6b918b8393378
  Stored in directory: /root/.cache/pip/wheels/80/ad/d4/9e1a0939f63331a3898f2a951a368bbf0d69f7b027cae4d66b
Successfully built vncorenlp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.8 MB/s et

# Config

In [2]:
import os
import random
import re
import ast

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

SEED = 42
MODEL_NAME = "vinai/phobert-base"  
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

USE_SMALL_DEBUG_SET = False   
SMALL_TRAIN_SIZE = 20000
SMALL_VAL_SIZE = 2000

torch.backends.cudnn.benchmark = True

Device: cuda


# Set seed, clean text

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


# Load train dataset

In [4]:
import gdown

file_id = "12bvIQP9kHpyf7MPPeQZFI60obPGOX4AY"
url = f"https://drive.google.com/uc?id={file_id}"
output_csv = "oreo_news_summarization_vi_train.csv"

gdown.download(url, output_csv, quiet=False)

print("File downloaded:", output_csv)


Downloading...
From (original): https://drive.google.com/uc?id=12bvIQP9kHpyf7MPPeQZFI60obPGOX4AY
From (redirected): https://drive.google.com/uc?id=12bvIQP9kHpyf7MPPeQZFI60obPGOX4AY&confirm=t&uuid=1c649819-ced5-423b-98ce-78f4280146dd
To: /kaggle/working/oreo_news_summarization_vi_train.csv
100%|██████████| 162M/162M [00:00<00:00, 167MB/s] 

File downloaded: oreo_news_summarization_vi_train.csv


# 

# Expand docs

In [5]:
df_doc = pd.read_csv(output_csv)
print("Raw shape:", df_doc.shape)
df_doc.head(3)

Raw shape: (28251, 4)


,content,sentences,labels,summary
0,Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoại ...,['Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoạ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4991882145404...",Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoại ...
1,"Suốt bao năm, để dòng tranh này không bị rơi v...","['Suốt bao năm , để dòng tranh này không bị rơ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.5041876435279846, ...",Ông Đạt Trần Thành là một trong những nghệ nhâ...
2,Khách qua phà mê mải ngắm sông như muốn quên đ...,['Khách qua phà mê mải ngắm sông như muốn quên...,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Những người thường ra bờ sông Sài Gòn để hóng ...


In [6]:
def explode_docs(df):
    texts = []
    labels = []
    for _, row in df.iterrows():
        try:
            sents = ast.literal_eval(row["sentences"])
            labs  = ast.literal_eval(row["labels"])
        except Exception:
            continue

        if not isinstance(sents, list) or not isinstance(labs, list):
            continue

        for s, l in zip(sents, labs):
            if not isinstance(s, str):
                continue
            s_clean = clean_text(s)
            if not s_clean:
                continue
            texts.append(s_clean)
            labels.append(float(l))  

    return pd.DataFrame({"text": texts, "label": labels})

df_sent = explode_docs(df_doc[:15000])
print("Tổng số câu:", len(df_sent))
print(df_sent["label"].value_counts(normalize=True))
df_sent.head()

Tổng số câu: 228879
label
0.000000    0.801970
1.000000    0.066245
0.500000    0.000402
0.250000    0.000192
0.333333    0.000039
              ...   
0.477725    0.000004
0.522275    0.000004
0.511027    0.000004
0.488973    0.000004
0.518729    0.000004
Name: proportion, Length: 29479, dtype: float64


,text,label
0,Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoại ...,0.0
1,"Tại buổi tiếp , Thủ tướng Nguyễn Xuân Phúc bày...",0.0
2,Ảnh : VGP / Quang Hiếu .,0.0
3,Thủ tướng Nguyễn Xuân Phúc đề nghị Bộ trưởng N...,0.0
4,Thủ tướng Nguyễn Xuân Phúc cũng đánh giá cao c...,0.0


# Train test split

In [7]:
set_seed(SEED)

train_df, val_df = train_test_split(
    df_sent,
    test_size=0.2,
    random_state=SEED,
    stratify=df_sent["label"] if df_sent["label"].nunique() == 2 else None,
)

if USE_SMALL_DEBUG_SET:
    train_df = train_df.sample(
        n=min(SMALL_TRAIN_SIZE, len(train_df)),
        random_state=SEED
    )
    val_df = val_df.sample(
        n=min(SMALL_VAL_SIZE, len(val_df)),
        random_state=SEED
    )
    print(f"[DEBUG] Train size={len(train_df)}, Val size={len(val_df)}")
else:
    print(f"Train size={len(train_df)}, Val size={len(val_df)}")


Train size=183103, Val size=45776


# Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print("Tokenizing train...")
train_encodings = tokenizer(
    train_df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
)

print("Tokenizing val...")
val_encodings = tokenizer(
    val_df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
)

train_labels = train_df["label"].tolist()
val_labels   = val_df["label"].tolist()


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train...
Tokenizing val...


# Dataset

In [9]:
class SentenceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }
        return item

train_dataset = SentenceDataset(train_encodings, train_labels)
val_dataset   = SentenceDataset(val_encodings, val_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

len(train_loader), len(val_loader)


(5722, 1431)

# Model

In [10]:
class PhoBERTClassifier(nn.Module):
    """
    PhoBERT + linear head nhị phân.
    Dùng CLS embedding (vị trí 0).
    """
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        hidden_size = self.phobert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :]  # (batch, hidden)
        x = self.dropout(cls_emb)
        logits = self.classifier(x).squeeze(-1)       # (batch,)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels)

        return loss, logits


# Training

In [ ]:
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch_idx):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_idx} [Train]",
        leave=False
    )

    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            loss, logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        avg_loss = total_loss / (step + 1)

        progress_bar.set_postfix(loss=f"{avg_loss:.4f}")

    return total_loss / max(len(loader), 1)



@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    all_labels = []
    all_preds = []

    progress_bar = tqdm(
        loader,
        desc="Eval",
        leave=False
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        loss, logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        total_loss += loss.item()
        total_batches += 1

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

        avg_loss = total_loss / total_batches
        progress_bar.set_postfix(val_loss=f"{avg_loss:.4f}")

    if total_batches == 0:
        return 0.0, 0.0

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    acc = (all_labels == all_preds).mean()
    avg_loss = total_loss / total_batches
    return avg_loss, acc

In [ ]:
set_seed(SEED)

model = PhoBERTClassifier(MODEL_NAME)
model.to(DEVICE)

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [
            p for n, p in model.named_parameters()
            if not any(nd in n for nd in no_decay)
        ],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [
            p for n, p in model.named_parameters()
            if any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters,
    lr=LR,
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = GradScaler()

print("Total steps:", total_steps, "| Warmup steps:", warmup_steps)


In [ ]:
best_val_loss = float("inf")
os.makedirs("checkpoints", exist_ok=True)
best_ckpt_path = "checkpoints/best_phobert_extractive.pt"

for epoch in range(1, EPOCHS + 1):
    print("=" * 60)
    print(f"Epoch {epoch}/{EPOCHS}")

    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, epoch
    )
    val_loss, val_acc = eval_one_epoch(model, val_loader)

    print(
        f"[Epoch {epoch}] "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt_path)
        print(f">>> Saved BEST model to {best_ckpt_path} (val_loss={val_loss:.4f})")

print("Training done.")
print("Best model path:", best_ckpt_path)

# Save

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base"
best_ckpt_path = "checkpoints/best_phobert_extractive.pt"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = PhoBERTClassifier(MODEL_NAME)
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()


# Evaluate

In [13]:
import py_vncorenlp
py_vncorenlp.download_model(save_dir='./')

from vncorenlp import VnCoreNLP
from rouge_score import rouge_scorer

rdr = VnCoreNLP("./VnCoreNLP-1.2.jar", annotators='wseg', max_heap_size='-Xmx2g')

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)

--2025-11-24 14:42:11--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 3.96M 7s
    50K .......... .......... .......... .......... ..........  0% 4.95M 6s
   100K .......... .......... .......... .......... ..........  0% 25.5M 4s
   150K .......... .......... .......... .......... ..........  0% 20.5M 4s
   200K .......... .......... .......... .......... ..........  0% 21.6M 3s
   250K .......... .......... .......... .......... ..........  1% 7.85M 3s
   300K .......... .......... .......... .......... ..........  1% 57.5M 3s
   350K ..

In [14]:
class PhoBERTClassifier(nn.Module):
    """
    PhoBERT + linear head nhị phân.
    Dùng CLS embedding (vị trí 0).
    """
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        hidden_size = self.phobert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :]  # (batch, hidden)
        x = self.dropout(cls_emb)
        logits = self.classifier(x).squeeze(-1)       # (batch,)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels)

        return loss, logits


In [15]:
import numpy as np

def split_sentences(text):
    if not isinstance(text, str):
        return []
    sents = rdr.tokenize(text)
    sents = [' '.join(sent) for sent in sents if sent]
    sents = [re.sub(r'_', ' ', sent) for sent in sents]
    return sents

def score_sentences_in_doc(sentences, model, tokenizer, max_len=128):
    """
    Trả về list xác suất (0–1) cho từng câu.
    """
    model.eval()
    device = next(model.parameters()).device

    all_probs = []
    for s in sentences:
        enc = tokenizer(
            s,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            _, logits = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                labels=None
            )
            prob = torch.sigmoid(logits).item()
        all_probs.append(prob)

    return all_probs

def build_extractive_summary(doc_text, model, tokenizer, top_k=3, max_len=128):
    """
    Từ 1 văn bản gốc => chọn top_k câu có xác suất cao nhất, ghép thành tóm tắt.
    """
    sents = split_sentences(doc_text)
    if not sents:
        return ""
    probs = score_sentences_in_doc(sents, model, tokenizer, max_len=max_len)

    # lấy index top_k câu quan trọng nhất
    idx = np.argsort(probs)[::-1][:top_k]
    idx_sorted = sorted(idx)  # giữ thứ tự gốc trong văn bản
    selected_sents = [sents[i] for i in idx_sorted]
    return " ".join(selected_sents)

import math

def build_extractive_summary_ratio(
    doc_text,
    model,
    tokenizer,
    keep_ratio=0.2,
    max_len=128,
    min_sentences=1,
    max_sentences=None,
):
    """
    Tóm tắt extractive theo % số câu:
      - keep_ratio: tỷ lệ câu giữ lại, ví dụ 0.2 = 20%
      - min_sentences: tối thiểu số câu giữ lại
      - max_sentences: tối đa số câu giữ lại (None = không giới hạn)
    """
    sents = split_sentences(doc_text)
    if not sents:
        return ""

    n = len(sents)

    k = max(min_sentences, int(math.ceil(n * keep_ratio)))
    if max_sentences is not None:
        k = min(k, max_sentences)
    k = min(k, n) 

    probs = score_sentences_in_doc(sents, model, tokenizer, max_len=max_len)

    idx = np.argsort(probs)[::-1][:k]
    idx_sorted = sorted(idx) 
    selected_sents = [sents[i] for i in idx_sorted]
    return " ".join(selected_sents)


In [16]:
import gdown

ref_file_id = "1RT5MuwNl6alaYmu6HPUP87NyFibK7SiJ"
ref_url = f"https://drive.google.com/uc?id={ref_file_id}"
ref_output = "oreo_news_summarization_vi_ref.csv"

gdown.download(ref_url, ref_output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1RT5MuwNl6alaYmu6HPUP87NyFibK7SiJ
To: /kaggle/working/oreo_news_summarization_vi_ref.csv
100%|██████████| 54.0M/54.0M [00:00<00:00, 168MB/s]


'oreo_news_summarization_vi_ref.csv'

In [17]:
weight_id = "1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-"
weight_url = f"https://drive.google.com/uc?id={weight_id}"
best_ckpt_path = "best_phobert_extractive.pt"
gdown.download(weight_url, best_ckpt_path, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-
From (redirected): https://drive.google.com/uc?id=1ph9GmQ0d_Zy79J8A3Ha6-z6zBf-9id0-&confirm=t&uuid=c76b20cf-8536-417e-a86d-e4d8c0de6753
To: /kaggle/working/best_phobert_extractive.pt
100%|██████████| 540M/540M [00:02<00:00, 190MB/s]  


'best_phobert_extractive.pt'

In [18]:
import pandas as pd

df_ref = pd.read_csv(ref_output)
print("Reference shape:", df_ref.shape)
print(df_ref.head())

Reference shape: (9450, 4)
                                             content  \
0  Hướng dẫn điều chỉnh giá hợp đồng xây dựng.\nQ...   
1  Vũ Thu Phương xuất hiện ở vị trí vedette. Ngườ...   
2  Nhắc đến cái tên Jenny McCarthy, người ta thườ...   
3  Thực tế, đến nay Bộ Thông tin và Truyền thông ...   
4  Theo trang tin quốc phòng militaryparitet (Nga...   

                                           sentences  \
0  ['Hướng dẫn điều chỉnh giá hợp đồng xây dựng ....   
1  ['Vũ Thu Phương xuất hiện ở vị trí vedette .',...   
2  ['Nhắc đến cái tên Jenny McCarthy , người ta t...   
3  ['Thực tế , đến nay Bộ Thông tin và Truyền thô...   
4  ['Theo trang tin quốc phòng militaryparitet ( ...   

                                              labels  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.48525065183639526,...   
1  [0.5235400795936584, 0.0, 0.0, 0.4764599204063...   
2  [0.0, 0.0, 0.4870632588863373, 0.0, 0.0, 0.512...   
3  [0.5070881247520447, 1.0, 0.4929119348526001, ...   
4  [0.4902909696102

In [19]:
def detok_for_bleu(text: str) -> str:
    text = re.sub(r"\s+([.!?,;:])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [20]:
import sacrebleu
DOC_COL = "content"   
REF_COL = "summary"   

def evaluate_rouge_bleu(
    df,
    model,
    tokenizer,
    doc_col,
    ref_col,
    top_k=3,
    keep_ratio=None,   
    max_docs=None
):
    """
    Duyệt từng doc:
      - sinh tóm tắt extractive bằng model
        + nếu keep_ratio != None: chọn theo % số câu
        + ngược lại: dùng top_k câu
      - so với summary tham chiếu bằng ROUGE + BLEU
    Trả về:
      - ROUGE trung bình (F1)
      - BLEU (corpus-level, sacrebleu)
    """
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    all_refs = []
    all_preds = []

    for i, row in df.iterrows():
        doc = str(row[doc_col])
        ref = clean_text(str(row[ref_col]))
        if not doc or not ref:
            continue

        if keep_ratio is not None:
            pred = clean_text(
                build_extractive_summary_ratio(  
                    doc,
                    model,
                    tokenizer,
                    keep_ratio=keep_ratio,
                    max_len=MAX_LEN
                )
            )
        else:
            pred = clean_text(
                build_extractive_summary(      
                    doc,
                    model,
                    tokenizer,
                    top_k=top_k,
                    max_len=MAX_LEN
                )
            )

        if not pred:
            continue

        s = scorer.score(ref, pred)
        for k in scores:
            scores[k].append(s[k].fmeasure)

        all_refs.append(ref)
        all_preds.append(pred)

        if max_docs is not None and len(scores["rouge1"]) >= max_docs:
            break

    rouge_avg = {k: (float(np.mean(v)) if v else 0.0) for k, v in scores.items()}

    if all_preds and all_refs:
        detok_preds = [detok_for_bleu(t) for t in all_preds]
        detok_refs  = [detok_for_bleu(t) for t in all_refs]
        bleu = sacrebleu.corpus_bleu(all_preds, [all_refs]).score
    else:
        bleu = 0.0

    metrics = {
        "rouge1": rouge_avg["rouge1"],
        "rouge2": rouge_avg["rouge2"],
        "rougeL": rouge_avg["rougeL"],
        "bleu": float(bleu),
    }
    return metrics

In [21]:
model = PhoBERTClassifier(MODEL_NAME)
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

avg_scores = evaluate_rouge_bleu(
    df_ref,
    model,
    tokenizer,
    doc_col=DOC_COL,
    ref_col=REF_COL,
    keep_ratio=0.25,    
    max_docs=5000
)

print("ROUGE F1 trung bình:")
for k, v in avg_scores.items():
    print(f"{k}: {v:.4f}")


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

ROUGE F1 trung bình:
rouge1: 0.6798
rouge2: 0.4930
rougeL: 0.4875
bleu: 29.8374


In [24]:
def preview_summaries(
    df,
    model,
    tokenizer,
    doc_col="content",
    ref_col="summary",
    num_samples=5,
    top_k=3,
    keep_ratio=None,
    max_len=128
):
    df_sample = df.sample(
        n=min(num_samples, len(df)),
        random_state=SEED
    ).reset_index(drop=True)

    for i, row in df_sample.iterrows():
        doc = str(row[doc_col])
        ref = str(row.get(ref_col, "") if ref_col in df.columns else "")

        pred = build_extractive_summary(
            doc,
            model,
            tokenizer,
            top_k=top_k,
            max_len=max_len,
        )
        if keep_ratio is not None:
            pred = clean_text(
                    build_extractive_summary_ratio(  
                        doc,
                        model,
                        tokenizer,
                        keep_ratio=keep_ratio,
                        max_len=MAX_LEN
                    )
                )

        print("=" * 100)
        print(f"[VĂN BẢN {i}]")
        print("-" * 100)

        max_chars_doc = 800
        print("VĂN BẢN GỐC:")
        print(doc[:max_chars_doc] + (" ..." if len(doc) > max_chars_doc else ""))
        print()

        if ref.strip():
            max_chars_ref = 400
            print("TÓM TẮT THAM CHIẾU:")
            print(ref[:max_chars_ref] + (" ..." if len(ref) > max_chars_ref else ""))
            print()

        max_chars_pred = 400
        print(f"TÓM TẮT EXTRACTIVE (top_k = {top_k} câu):")
        print(pred[:max_chars_pred] + (" ..." if len(pred) > max_chars_pred else ""))
        print()


In [26]:
preview_summaries(
    df=df_ref,
    model=model,
    tokenizer=tokenizer,
    doc_col=DOC_COL,   # "content"
    ref_col=REF_COL,   # "summary"
    num_samples=3,
    keep_ratio=0.3,
    max_len=MAX_LEN
)


[VĂN BẢN 0]
----------------------------------------------------------------------------------------------------
VĂN BẢN GỐC:
Tập đoàn Công nghiệp quốc phòng Izhmash của Nga bắt đầu thử nghiệm loại súng trường tấn công mới AK-200 nhằm thay thế phiên bản súng AK-100 mà quân đội Nga đang sử dụng. Tính năng kỹ, chiến thuật của AK-200 tương đương với các loại vũ khí có tính năng tương tự và tối tân nhất trên thế giới hiện nay. AK-200 được thiết kế theo dạng mô-đun, cho phép thay đổi nhanh cấu hình, có thể tương thích với các loại thiết bị hỗ trợ và các hệ thống vũ khí khác. Hiệu quả tác chiến của AK-200 cao hơn từ 40% đến 50% so với thế hệ AK trước. Đặc điểm nổi bật của AK-200 là cơ cấu ngắm hở, hộp đạn có dung lượng lớn chứa từ 50 đến 60 viên và có khả năng thay cỡ đạn nhanh chóng bằng cách thay nòng súng. AK-200 còn có giá để lắp các thiết bị phụ trợ như máy ngắm, bộ chỉ thị mục tiêu la-de và đèn pin. Tính năng và  ...

TÓM TẮT THAM CHIẾU:
Tập đoàn Công nghiệp quốc phòng Izhmash của Nga 